In [ ]:
import torch
import datasets
from transformers import AutoTokenizer

In [ ]:
import numpy as np

In [ ]:
import copy

In [ ]:
#from lm_polygraph.generation_metrics import AlignScore

# Define functions

In [ ]:
def label_data(ds, scorer, gen_field="reply", target_field="correct_answer"):
    splits = list(ds.keys())
    for split in splits:
        greedy = ds[split][gen_field]
        target = ds[split][target_field]
        scores = scorer({"greedy_texts": greedy}, target)
        ds[split] = ds[split].add_column("alignscore", scores)
    return ds

In [ ]:
def extract_tokens_of_reply(dataset, tokenizer):
    inpt_texts = dataset["question"]
    question_tokens = tokenizer.apply_chat_template(
            [[{"role": "user", "content": e}] for e in inpt_texts],
            tokenize=True,
            add_generation_prompt=True,
        )
    greedy_tokens = []
    inpt_ids = dataset["input_ids"]
    for i in range(len(dataset)): # This whole thing is necessary because we need tokens generated by LLM not how reply will be tokenized
        tokens_with_special = inpt_ids[i][len(question_tokens[i]):]
        greedy_tokens.append(
            [t for t in tokens_with_special if t not in tokenizer.all_special_ids]
        )
    
    return greedy_tokens

In [ ]:
def ue_label_mapping(label, ue_label_threshold):
    return 1 if label > ue_label_threshold else 0

In [ ]:
def rebuild_dataset(dataset, ue_label_threshold=0.54):
    ds = copy.deepcopy(dataset)
    splits = list(ds.keys())
    for split in splits:
        # build claims
        greedy_tokens = extract_tokens_of_reply(ds[split], tokenizer)
        claims = []
        unc_labels = []
        unc_labels_orig = []
        verified = []
        for idx, el in enumerate(ds[split]):
            claims.append([{
                "aligned_token_ids": list(range(len(greedy_tokens[idx]))),
                "claim_text": el["reply"],
                "sentence": el["reply"],
            }])
            unc_labels_orig.append(torch.concatenate([torch.tensor([-100]*(len(el["input_ids"]) - len(greedy_tokens[idx]))), torch.tensor([el["alignscore"]]*len(greedy_tokens[idx]))], dim=-1).numpy().tolist())
            unc_labels.append(torch.concatenate([torch.tensor([-100]*(len(el["input_ids"]) - len(greedy_tokens[idx]))), torch.tensor([ue_label_mapping(el["alignscore"], ue_label_threshold)]*len(greedy_tokens[idx]))], dim=-1).numpy().tolist())
            verified.append([ue_label_mapping(el["alignscore"], ue_label_threshold)])
        ds[split] = ds[split].add_column("claims", claims)
        ds[split] = ds[split].add_column("verified", verified)
        ds[split] = ds[split].add_column("uncertainty_labels_original", unc_labels_orig)
        ds[split] = ds[split].add_column("uncertainty_labels", unc_labels)
    return ds
        # TODO: tokenize reply, calc tokens, add array
        # build verified
        # TODO: add length 1 array with label
        # build unc label and orig - map to each token in input_ids

In [ ]:
def rebuild_data_split_from_polygraph(data, ue_label_threshold=0.54, instruct=True):
    source_data = data["stats"]
    source_data_scores_test = data["gen_metrics"][('sequence', 'AlignScore')]
    keys_mapping = {'input_texts': 'question', 'target_texts': 'answer', 'greedy_texts': 'reply', 'greedy_tokens': 'input_ids'}
    source_data_scores = source_data_scores_test
    new_data = {}
    new_inps = []
    new_labels = []
    new_labels_bin = []
    claims = []
    def ue_label_mapping(label, ue_label_threshold):
        return 1 if label > ue_label_threshold else 0
    for key in keys_mapping.keys():
        if "greedy_tokens" in key:
            inpt_texts = source_data["input_texts"]
            # concat question ids with greedy ids for compatibility with luq code
            if instruct:
                formatted_texts = []
                for chat in inpt_texts:
                    chat = [{"role": "user", "content": chat}]
                    formatted_chat = tokenizer.apply_chat_template(
                        chat, add_generation_prompt=True, tokenize=False
                    )
                    formatted_texts.append(formatted_chat)
                batch = tokenizer(formatted_texts, padding=False, return_tensors=None, add_special_tokens=False)
            else:
                batch = tokenizer(inpt_texts, padding=False, return_tensors=None)
            for idx, outp in enumerate(source_data[key]):
                inp = batch["input_ids"][idx]
                new_inps.append(torch.concatenate([torch.tensor(inp), torch.tensor(outp)], dim=-1))
                new_labels.append(torch.concatenate([torch.tensor([-100]*len(inp)), torch.tensor([source_data_scores[idx]]*len(outp))], dim=-1))
                new_labels_bin.append(torch.concatenate([torch.tensor([-100]*len(inp)), torch.tensor([ue_label_mapping(source_data_scores[idx], ue_label_threshold)]*len(outp))], dim=-1))
                claims.append([{
                    "aligned_token_ids": list(range(len(outp))),
                    "claim_text": source_data["greedy_texts"][idx],
                    "sentence": source_data["greedy_texts"][idx],
                }])
            new_data["input_ids"] = new_inps
            new_data["claims"] = claims
            new_data["greedy_ids"] = source_data[key]
            new_data["uncertainty_labels_original"] = new_labels
            new_data["uncertainty_labels"] = new_labels_bin
            new_data["verified"] = [[el[-1].item()] for el in new_labels_bin]
            new_data["alignscore"] = source_data_scores
        else:
            new_data[keys_mapping[key]] = source_data[key]
    rebuild_data = datasets.Dataset.from_dict(new_data)
    return rebuild_data

# Functions for target and claim matching

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, KFold

In [ ]:
import re
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import List, Dict, Tuple, Any
import os
from scipy.optimize import linear_sum_assignment
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
# matchers for diagnoses - seq level (by dict), claim level (by dict), and neural
class MedicalMatcher:
    def __init__(self, json_data):
        self.pairs = json_data
    
    def match(self, diagnosis1, diagnosis2):
        diagnosis1 = diagnosis1.lower()
        diagnosis2 = diagnosis2.lower()
        
        key1 = f"{diagnosis1}|{diagnosis2}"
        key2 = f"{diagnosis2}|{diagnosis1}"
        
        if key1 in self.pairs:
            return self.pairs[key1][0]
        if key2 in self.pairs:
            return self.pairs[key2][0]
        
        words1 = set(diagnosis1.split())
        words2 = set(diagnosis2.split())
        intersection = len(words1.intersection(words2))
        union = len(words1.union(words2))
        return intersection / union if union > 0 else 0.0

class MedicalMatcherClaim:
    def __init__(self, json_data):
        self.pairs = json_data
    
    def match(self, diagnosis1, diagnosis2):
        diagnosis1 = diagnosis1.lower()
        diagnosis2 = diagnosis2.lower()
        
        key1 = f"{diagnosis1}|{diagnosis2}"
        key2 = f"{diagnosis2}|{diagnosis1}"

        score_map = {True: 1, False: 0}
        if key1 in self.pairs:
            return score_map[self.pairs[key1][1]]
        if key2 in self.pairs:
            return score_map[self.pairs[key2][1]]
        return 0

class MedicalMatcherClaimNeural:
    def __init__(self, nli_model_path="./med_data/distilled-dm-model"):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(nli_model_path)
        model = AutoModelForSequenceClassification.from_pretrained(nli_model_path)
        model.to(device)
        model.eval()
        self.device = device
        self.model = model
        self.tokenizer = tokenizer
    
    def match(self, diagnosis1, diagnosis2):
        diagnosis1 = diagnosis1.lower()
        diagnosis2 = diagnosis2.lower()
        
        key1 = f"{diagnosis1}|{diagnosis2}"
        key2 = f"{diagnosis2}|{diagnosis1}"
        inputs = self.tokenizer(
            diagnosis1, diagnosis2,
            padding=True, truncation=True, max_length=128,
            return_tensors="pt"
        ).to(self.device)
        all_probs = []
        with torch.no_grad():
            logits = self.model(**inputs).logits
            probs = torch.softmax(logits, dim=1)
            all_probs.extend(probs[:, 1].cpu().tolist())
        y_scores = np.array(all_probs)
        # print(diagnosis1, diagnosis2, y_scores)
        return 1 if y_scores[0] > 0.5 else 0


In [ ]:
def preprocess_prediction_fixed_format(prediction, lite_train_split=False):
    
    if not prediction:
        return []
    
    if isinstance(prediction, list):
        if len(prediction) > 0:
            prediction_text = prediction[0]
        else:
            return []
    else:
        prediction_text = str(prediction)
    if "<diag_diag>" in prediction_text:
        #lines = prediction_text.split('<DIAG>')
        buf_lines = prediction_text.split('<diag_diag>')
        if lite_train_split:
            splitted_lines = [el.split('</diag_diag>')[0] for el in buf_lines if len(el) != 0]
        else:
            splitted_lines = [el.replace('</diag_diag>', "") for el in buf_lines if len(el) != 0]
        lines = []
        for el in splitted_lines:
            lines.append(el)
    else:
        lines = prediction_text.split(';')
    diagnoses = []
    diagnoses_orig = []
    
    for line in lines:
        line = line.strip()
        match = re.match(r'^(\d+\.\s*)?([^\(\)\\n]+?)(?=\\n|\n|$|\(|\.\s|$)', line)
        if match:
            diagnosis = match.group(2).strip().lower()
            diagnosis_orig = match.group(2).strip()
            if (len(diagnosis) > 3 and 
                not diagnosis.startswith('эти') and
                not diagnosis.startswith('наиболее') and
                '...' not in diagnosis):
                diagnoses.append(diagnosis)
                diagnoses_orig.append(diagnosis_orig)
    
    return diagnoses[:3], diagnoses_orig[:3]

def preprocess_target_fixed_format(target):
    
    if not target:
        return []

    lines = target.split(',')
    diagnoses = []
    for line in lines:
        line = line.strip()
        match = re.match(r'^(\d+\.\s*)?([^\(\)\\n]+?)(?=\\n|\n|$|\(|\.\s|$)', line)
        if match:
            diagnosis = match.group(2).strip().lower()
            
            if (len(diagnosis) >= 3 and 
                not diagnosis.startswith('эти') and
                not diagnosis.startswith('наиболее') and
                '...' not in diagnosis):
                diagnoses.append(diagnosis)
    return diagnoses#[:3]

def preprocess_target_enlarged_train(target):
    
    if not target:
        return []
    
    lines = target.split('\n')
    diagnoses = []
    for line in lines:
        line = line.strip()
        match = re.match(r'^(\d+\.\s*)?([^\(\)\\n]+?)(?=\\n|\n|$|\(|\.\s|$)', line)
        if match:
            diagnosis = match.group(2).strip().lower()
            
            if (len(diagnosis) >= 3 and 
                not diagnosis.startswith('эти') and
                not diagnosis.startswith('наиболее') and
                '...' not in diagnosis):
                diagnoses.append(diagnosis)
    return diagnoses#[:3]

def hungarian_matching(pred_diagnoses, true_diagnoses, matcher):
    if not pred_diagnoses or not true_diagnoses:
        return 0.0
        
    similarity_matrix = np.zeros((len(pred_diagnoses), len(true_diagnoses)))
    for i, pred in enumerate(pred_diagnoses):
        for j, true in enumerate(true_diagnoses):
            similarity_matrix[i, j] = matcher.match(pred, true)
    
    row_ind, col_ind = linear_sum_assignment(-similarity_matrix)
    return np.mean(similarity_matrix[row_ind, col_ind])

def calculate_medical_score(item, matcher):
    
    if 'prediction' not in item or 'target' not in item:
        return 0.0
    pred_diagnoses = preprocess_prediction(item['prediction'])
    target_diagnoses = [diag.lower() for diag in item['target']]
    
    if not pred_diagnoses or not target_diagnoses:
        return 0.0
    
    return round(hungarian_matching(pred_diagnoses, target_diagnoses, matcher), 4)

def relaxed_em_matching(pred_diagnoses, true_diagnoses, matcher):
    if not pred_diagnoses or not true_diagnoses:
        return 0.0
        
    similarity_matrix = np.zeros((len(pred_diagnoses), len(true_diagnoses)))
    #print(similarity_matrix.shape)
    for i, pred in enumerate(pred_diagnoses):
        for j, true in enumerate(true_diagnoses):
            similarity_matrix[i, j] = matcher.match(pred, true)

    #print(similarity_matrix)
    #print(np.max(similarity_matrix, axis=1))
    #row_ind, col_ind = linear_sum_assignment(-similarity_matrix)
    return np.max(similarity_matrix, axis=1)

def calculate_em_claim(item, matcher, lite_train_split=False):
    
    if 'prediction' not in item or 'target' not in item:
        return 0.0
    #print(item['prediction'])
    #print(item['target'])
    pred_diagnoses, pred_diagnoses_orig = preprocess_prediction_fixed_format(item['prediction'], lite_train_split)
    #print(pred_diagnoses)
    target_diagnoses = preprocess_target_fixed_format(item['target'])#[diag.lower() for diag in item['target']]
    if not pred_diagnoses or not target_diagnoses:
        return np.array([0]), [item['prediction']]
    
    return relaxed_em_matching(pred_diagnoses, target_diagnoses, matcher), pred_diagnoses_orig

def calculate_em_claim_enlarged_train(item, matcher, lite_train_split=False):
    
    if 'prediction' not in item or 'target' not in item:
        return 0.0
    #print(item['prediction'])
    #print(item['target'])
    pred_diagnoses, pred_diagnoses_orig = preprocess_prediction_fixed_format(item['prediction'], lite_train_split)
    #print(pred_diagnoses)
    target_diagnoses = preprocess_target_enlarged_train(item['target'])#[diag.lower() for diag in item['target']]
    if not pred_diagnoses or not target_diagnoses:
        return np.array([0]), [item['prediction']]
    
    return relaxed_em_matching(pred_diagnoses, target_diagnoses, matcher), pred_diagnoses_orig


def parse_log_file(log_content: str, matcher=None) -> List[Dict]:
    results = []
    current_batch = {}
    current_estimations = {}
    current_matching_scores = []  
    in_estimations = False
    in_generation_metrics = False

    input_re = re.compile(r"input_texts: \['(.*?)'\]")
    target_re = re.compile(r"target_texts: \['(.*?)'\]")
    greedy_re = re.compile(r"greedy_texts: \['(.*?)'\]")
    estimator_re = re.compile(r"\('sequence', '(.*?)'\): \[(.*?)\]")
    
    result_re = re.compile(r"RESULT: (?:True|False) \(([\d.]+)\)")

    for line in log_content.split('\n'):
        if "================================================== NEW BATCH ==================================================" in line:
            
            if current_batch:
                
                if current_matching_scores:
                    
                    current_estimations['matching_scores'] = current_matching_scores
                    current_estimations['max_matching_score'] = max(current_matching_scores)
                    current_estimations['min_matching_score'] = min(current_matching_scores)
                    current_estimations['minmax_matching_score'] = 1 - (max(current_matching_scores) - min(current_matching_scores))
                    medical_score = calculate_medical_score(current_batch, matcher)
                    current_estimations['medical_match'] = medical_score
                
                if current_estimations:
                    current_batch['estimations'] = current_estimations
                
                results.append(current_batch)
            
            
            current_batch = {}
            current_estimations = {}
            current_matching_scores = []  
            in_estimations = False
            in_generation_metrics = False
            continue  

        
        if "RESULT:" in line:
            match = result_re.search(line)
            if match:
                score = float(match.group(1))
                current_matching_scores.append(score)

        if "input_texts" in line:
            match = input_re.search(line)
            if match:
                current_batch['input'] = match.group(1)

        if "target_texts" in line:
            match = target_re.search(line)
            if match:
                current_batch['target'] = [d.strip() for d in match.group(1).split(',') if d.strip()]

        if "greedy_texts" in line:
            match = greedy_re.search(line)
            if match:
                current_batch['prediction'] = [d.strip() for d in match.group(1).split(',') if d.strip()]

        if "Estimations:" in line:
            in_estimations = True
            in_generation_metrics = False
            continue

        if "Generation metrics:" in line:
            in_estimations = False
            in_generation_metrics = True
            continue

        if (in_estimations or in_generation_metrics) and line.strip():
            match = estimator_re.search(line)
            if match:
                estimator_name = match.group(1)
                value_str = match.group(2)
                try:
                    if ',' in value_str:
                        values = [float(v) for v in value_str.split(',')]
                        value = values[0] if values else 0.0
                    else:
                        value = float(value_str)
                    current_estimations[estimator_name] = value
                except ValueError:
                    continue

    
    if current_batch:
        
        if current_matching_scores:
            current_estimations['matching_scores'] = current_matching_scores
            current_estimations['max_matching_score'] = max(current_matching_scores)
            current_estimations['min_matching_score'] = min(current_matching_scores)
            current_estimations['minmax_matching_score'] = 1 - (max(current_matching_scores) - min(current_matching_scores))
        
            medical_score = calculate_medical_score(current_batch, matcher)
            current_estimations['medical_match'] = medical_score
        
        if current_estimations:
            current_batch['estimations'] = current_estimations
        
        results.append(current_batch)

    return results

In [ ]:
# TODO:
# build claim-level for Max with Lite approx
# seq-level for MAX with Lite approx
# claim-level for Lite
# seq-level for Lite

In [ ]:
base_file_path = "../med_data_paper/"

# [PAPER, Max - claim] Prepare claim-level data in the original format - now with fixed demographic

In [ ]:
# load data
small_orig_data_giga_big = datasets.load_dataset("AIRI-NLP/med_f_resp", download_mode=datasets.DownloadMode.FORCE_REDOWNLOAD)
big_orig_data_giga_big = datasets.load_dataset("AIRI-NLP/med_train_f_resp", download_mode=datasets.DownloadMode.FORCE_REDOWNLOAD)

In [ ]:
def transform_to_base_csv(ds, save_path, is_synth_answer=False):
    if is_synth_answer:
        col_mapping = {"dialogue": "input", "deepseek_response": "target", "model_response": "prediction"}
    else:
        col_mapping = {"dialogue": "input", "ans_str": "target", "model_response": "prediction"}
    buf = ds.to_pandas()
    buf = buf.rename(col_mapping, axis=1)
    buf.to_csv(save_path, index=False)

In [ ]:
# save base csv in the legacy format for future processing

In [ ]:
transform_to_base_csv(small_orig_data_giga_big["train"], save_path=base_file_path + "medical_data_ordered_giga_big_fixed_format_holdout_set_intro_fix.csv", is_synth_answer=False)

In [ ]:
transform_to_base_csv(small_orig_data_giga_big["test"], save_path=base_file_path+ "medical_data_ordered_giga_big_fixed_format_intro_fix.csv", is_synth_answer=False)

In [ ]:
transform_to_base_csv(big_orig_data_giga_big["test"], save_path=base_file_path + "medical_data_ordered_max_1500_giga_big_fixed_format_intro_fix.csv", is_synth_answer=True)

# GigaBig, Prepare claim-level data (orig format) - match with dict, test set

In [ ]:
# todo: split each answer on claims (and target)
# match each answer claim with target

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
with open(base_file_path + "artifacts/pair-match.json", 'r', encoding='utf-8') as f:
    matcher_data = json.load(f)
medical_matcher = MedicalMatcherClaim(matcher_data)

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
# load tokenizer from approx. model
tok = AutoTokenizer.from_pretrained("/home/jovyan/rvanova/giga1/Lite_v2/hf")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:-1] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        claim_tokens = tok(str(text_claim))["input_ids"][1:-1]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + "uhead_datasets/gigachat_big_with_lite/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.load_from_disk(base_file_path + "uhead_datasets/gigachat_big_with_lite/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.DatasetDict({"train": buf_dataset, "test": buf_dataset.select([0])})
buf_dataset.save_to_disk(base_file_path + "uhead_datasets/gigachat_big_with_lite/med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix_full_train")

In [ ]:
x, y = np.arange(len(new_dataset)), [el[0] for el in new_dataset["verified"]]
skf = KFold(n_splits=5, random_state=None)
skf.get_n_splits(x)
for i, (train, test) in enumerate(skf.split(x, y)):
    print(f"Fold {i}:")
    print(f"  Train:  index={train}")
    print(f"  Test:  index={test}")
    print(list(test))
    med_dataset = datasets.DatasetDict({
        "train": new_dataset.select(train),
        "test": new_dataset.select(test),
    })
    med_dataset.save_to_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix_cv_{i}")
#train, test = train_test_split(x, test_size=0.2, stratify=y, random_state=35)


In [ ]:
# inspect dataset stats - estimate base performance scores

In [ ]:
for idx in range(5):
    buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    t_labs = [el for el in buf_med_data["test"]["verified"]]
    t_labs_flat = []
    for el in t_labs:
        t_labs_flat += el
    print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
    print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

## Do the same for the small holdout set

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_holdout_set_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
nli_model_path = base_file_path + "artifacts/distilled-dm-model"
medical_matcher = MedicalMatcherClaimNeural(nli_model_path)

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
tok = AutoTokenizer.from_pretrained("/home/jovyan/rvanova/giga1/Lite_v2/hf")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:-1] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        claim_tokens = tok(str(text_claim))["input_ids"][1:-1]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims_neural_intro_fix.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_holdout_set_neural_intro_fix")

In [ ]:
new_dataset_v1 = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_holdout_set_neural_intro_fix")
new_dataset_v2 = datasets.DatasetDict({"train": new_dataset_v1, "test": new_dataset_v1})
new_dataset_v2.save_to_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_holdout_set_neural_intro_fix_test")

# Prepare claim-level data for 1.5k for big model (orig format)

In [ ]:
path = base_file_path + "medical_data_ordered_max_1500_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
nli_model_path = base_file_path + "artifacts/distilled-dm-model"
medical_matcher = MedicalMatcherClaimNeural(nli_model_path)

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim_enlarged_train(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
tok = AutoTokenizer.from_pretrained("/home/jovyan/rvanova/giga1/Lite_v2/hf")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:-1] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        if "1. " in new_data["reply"][idx]:
            claim_tokens = tok(" " + str(text_claim))["input_ids"][1:-1]            
        else:
            claim_tokens = tok(str(text_claim))["input_ids"][1:-1]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_big_enlarged_train_fixed_format_with_claims_intro_fix.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/raw_med_data_relaxed_em_target_claim_enlarged_train_fixed_format_intro_fix")

In [ ]:
new_dataset = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/raw_med_data_relaxed_em_target_claim_enlarged_train_fixed_format_intro_fix")

In [ ]:
for idx in range(5):
    med_dataset_test = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    med_dataset_test["train"] = datasets.concatenate_datasets([new_dataset, med_dataset_test["train"]])
    med_dataset_test.save_to_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")

In [ ]:
# split on the same train and test, as before (for med match)

In [ ]:
buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_0")

In [ ]:
for idx in range(5):
    buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    t_labs = [el for el in buf_med_data["test"]["verified"]]
    t_labs_flat = []
    for el in t_labs:
        t_labs_flat += el
    #print(t_labs, el)
    print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
    print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

In [ ]:
for idx in range(5):
    buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    t_labs = [el for el in buf_med_data["train"]["verified"]]
    t_labs_flat = []
    for el in t_labs:
        t_labs_flat += el
    print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
    print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

### Prepare dataset for hp - test - all 260 samples, other - train and valid

In [ ]:
new_dataset = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/raw_med_data_relaxed_em_target_claim_enlarged_train_fixed_format_intro_fix")

In [ ]:
for idx in range(5):
    med_dataset_test = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    if idx == 0:
        buf_test = med_dataset_test["test"]
    else:
        buf_test = datasets.concatenate_datasets([buf_test, med_dataset_test["test"]])

In [ ]:
buf_test

In [ ]:
# split train on train and val
train_ids = np.arange(len(new_dataset))
train_ids, val_ids = train_test_split(train_ids, test_size=0.1, random_state=43)

In [ ]:
len(train_ids), len(val_ids)

In [ ]:
data_with_val = {
    "train": new_dataset.select(train_ids),
    "validation": new_dataset.select(val_ids),
    "test": buf_test,
}
data_with_val = datasets.DatasetDict(data_with_val)

In [ ]:
data_with_val.save_to_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")

# [PAPER, Lite - claim] Prepare claim-level data in the original format - now with fixed demographic

In [ ]:
def parse_log_file_no_target(log_content: str) -> List[Dict]:
    results = []
    current_batch = {}
    current_estimations = {}
    current_matching_scores = []  
    in_estimations = False
    in_generation_metrics = False

    input_re = re.compile(r"input_texts: \['(.*?)'\]")
    target_re = re.compile(r"target_texts: \['(.*?)'\]")
    greedy_re = re.compile(r"greedy_texts: \['(.*?)'\]")
    estimator_re = re.compile(r"\('claim', '(.*?)'\): \[(.*?)\]")
    
    result_re = re.compile(r"RESULT: (?:True|False) \(([\d.]+)\)")
    prev_line = ""

    for line in log_content.split('\n'):
        if "================================================== NEW BATCH ==================================================" in line:
            
            if current_batch:
                
                if current_matching_scores:
                    
                    current_estimations['matching_scores'] = current_matching_scores
                    current_estimations['max_matching_score'] = max(current_matching_scores)
                    current_estimations['min_matching_score'] = min(current_matching_scores)
                    current_estimations['minmax_matching_score'] = 1 - (max(current_matching_scores) - min(current_matching_scores))
                    #medical_score = calculate_medical_score(current_batch, matcher)
                    #current_estimations['medical_match'] = medical_score
                
                if current_estimations:
                    current_batch['estimations'] = current_estimations
                
                results.append(current_batch)
            
            
            current_batch = {}
            current_estimations = {}
            current_matching_scores = []  
            in_estimations = False
            in_generation_metrics = False
            continue  

        
        if "RESULT:" in line:
            match = result_re.search(line)
            if match:
                score = float(match.group(1))
                current_matching_scores.append(score)

        if "input_texts" in line:
            match = input_re.search(line)
            if match:
                current_batch['input'] = match.group(1)

        if "target_texts" in line:
            match = target_re.search(line)
            if match:
                current_batch['target'] = [d.strip() for d in match.group(1).split(',') if d.strip()]

        if "greedy_texts" in line:
            match = greedy_re.search(line)
            greedy_text = line.split("greedy_texts:")[1].split("[")[1].split("]")[0]
            current_batch['greedy_text'] = greedy_text
            if match:
                current_batch['prediction'] = [d.strip() for d in match.group(1).split(',') if d.strip()]

        if "Estimations:" in line:
            in_estimations = True
            in_generation_metrics = False
            continue

        if "Generation metrics:" in line:
            in_estimations = False
            in_generation_metrics = True
            continue

        if (in_estimations or in_generation_metrics) and line.strip():
            match = estimator_re.search(line)
            if match:
                estimator_name = match.group(1)
                value_str = match.group(2)
                try:
                    #if ',' in value_str:
                    values = [float(v) for v in value_str.split(',')]
                    #value = values[0] if values else 0.0
                    #else:
                    #    value = float(value_str)
                    current_estimations[estimator_name] = values
                except ValueError:
                    continue
        prev_line = line

    
    if current_batch:
        
        if current_matching_scores:
            current_estimations['matching_scores'] = current_matching_scores
            current_estimations['max_matching_score'] = max(current_matching_scores)
            current_estimations['min_matching_score'] = min(current_matching_scores)
            current_estimations['minmax_matching_score'] = 1 - (max(current_matching_scores) - min(current_matching_scores))
        
            medical_score = calculate_medical_score(current_batch, matcher)
            current_estimations['medical_match'] = medical_score
        
        if current_estimations:
            current_batch['estimations'] = current_estimations
        
        results.append(current_batch)

    return results

In [ ]:
# here load from manager
#sample_data = torch.load("/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/AIRI-NLP/med_f/2026-02-06/16-53-32/ue_manager_seed1", weights_only=False)
sample_data = torch.load("/home/jovyan/gkuzmin/llm_ue/med_ue_polygraph/med_polygraph/workdir/output/qa/home/jovyan/rvanova/giga1/Lite_v2/hf/AIRI-NLP/med_f/2026-02-06/14-57-55/ue_manager_seed1", weights_only=False)

In [ ]:
# test set
log_file = '/home/jovyan/gkuzmin/llm_ue/med_ue_polygraph/med_polygraph/workdir/output/qa/home/jovyan/rvanova/giga1/Lite_v2/hf/AIRI-NLP/med_f/2026-02-06/14-57-55/polygraph_eval.log'
# holdout set
#log_file = '/home/jovyan/gkuzmin/llm_ue/med_ue_polygraph/med_polygraph/workdir/output/qa/home/jovyan/rvanova/giga1/Lite_v2/hf/AIRI-NLP/med_f/2026-02-06/16-39-14/polygraph_eval.log'
# train set
#log_file = '/home/jovyan/gkuzmin/llm_ue/med_ue_polygraph/med_polygraph/workdir/output/qa/home/jovyan/rvanova/giga1/Lite_v2/hf/AIRI-NLP/med_train_f/2026-02-06/17-03-57/polygraph_eval.log'
# test set with new NLI model
log_file = '/home/jovyan/gkuzmin/llm_ue/med_ue_polygraph/med_polygraph/workdir/output/qa/home/jovyan/rvanova/giga1/Lite_v2/hf/AIRI-NLP/med_f/2026-02-10/15-33-52/polygraph_eval.log'
# test set with old NLI model fix + new NLI on crossencoder
log_file = '/home/jovyan/gkuzmin/llm_ue/med_ue_polygraph/med_polygraph/workdir/output/qa/home/jovyan/rvanova/giga1/Lite_v2/hf/AIRI-NLP/med_f/2026-02-10/18-58-41/polygraph_eval.log'
# giga-max, BB, base NLI
log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/workdir/output/qa/Sber/GigaChat-Max-V2/AIRI-NLP/med_f/2026-02-12/10-15-38/polygraph_eval.log'
# giga-max, BB, new med NLI, some new setting - check if work
log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/workdir/output/qa/Sber/GigaChat-Max-V2/AIRI-NLP/med_f/2026-02-12/13-38-33/polygraph_eval.log'
# giga-max, BB, base NLI, some new setting - check if work
log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/workdir/output/qa/Sber/GigaChat-Max-V2/AIRI-NLP/med_f/2026-02-12/19-04-42/polygraph_eval.log'



with open(log_file, 'r', encoding='utf-8') as f:
    log_content = f.read()

results = parse_log_file_no_target(log_content)

In [ ]:
results[0]

In [ ]:
res_dict = {}
for key in results[0].keys():
    if key in ["prediction"]:
        res_dict[key] = [res[key][0] if key in res.keys() else "" for res in results]
    elif key in ["estimations"]:
        for inner_key in results[0][key].keys():
            if inner_key not in ["matching_scores", 'max_matching_score', 'min_matching_score', 'minmax_matching_score']:
                res_dict[f"{key}_{inner_key}"] = [res[key][inner_key] for res in results]
    else:
        res_dict[key] = [res[key] for res in results]

In [ ]:
for idx, res in enumerate(results):
    if idx == 18:
        print(res["prediction"])
    if "prediction" not in res.keys():
        #print(res)
        #print(res.keys())
        print(idx)

In [ ]:
res_df = pd.DataFrame.from_dict(res_dict)

In [ ]:
import copy
bse_nli_res_df = copy.deepcopy(res_df)

In [ ]:
np.array(res_df["prediction"] == bse_nli_res_df["prediction"])

In [ ]:
res_df.head(20)

In [ ]:
res_df.head(20)

In [ ]:
#res_df.to_csv("../med_data_paper/med_diag_lite_with_ue_test_old_nli_fix.csv", index=False)#[3]
#res_df.to_csv("../med_data_paper/med_diag_max_with_ue_test_new_nli.csv", index=False)#[3]
res_df.to_csv("../med_data_paper/med_diag_max_with_ue_test_old_nli.csv", index=False)#[3]

## Load new test with UE for BB GigaMax

In [ ]:
log_file_med_nli = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/workdir/output/qa/Sber/GigaChat-Max-V2/AIRI-NLP/med_f/2026-02-12/13-38-33/polygraph_eval.log'
# giga-max, BB, base NLI, some new setting - check if work
log_file_base_nli = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/workdir/output/qa/Sber/GigaChat-Max-V2/AIRI-NLP/med_f/2026-02-12/19-04-42/polygraph_eval.log'
# [DEBUG ONLY], paper res uncommmented. check mathes for initial v1 exps - where UE normal, but generations are strange
#log_file_med_nli = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/workdir/output/qa/Sber/GigaChat-Max-V2/AIRI-NLP/med_f/2026-02-12/08-03-17/polygraph_eval.log'
# giga-max, BB, base NLI, some new setting - check if work
#log_file_base_nli = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/workdir/output/qa/Sber/GigaChat-Max-V2/AIRI-NLP/med_f/2026-02-12/10-15-38/polygraph_eval.log'


with open(log_file_med_nli, 'r', encoding='utf-8') as f:
    log_content = f.read()

results = parse_log_file_no_target(log_content)

with open(log_file_base_nli, 'r', encoding='utf-8') as f:
    log_content = f.read()

results_base = parse_log_file_no_target(log_content)

In [ ]:
print(log_content[:40000])

In [ ]:
res_dict = {}
for key in results[0].keys():
    if key in ["prediction"]:
        res_dict[key] = [res[key][0] if key in res.keys() else "" for res in results]
    elif key in ["estimations"]:
        for inner_key in results[0][key].keys():
            if inner_key not in ["matching_scores", 'max_matching_score', 'min_matching_score', 'minmax_matching_score']:
                res_dict[f"{key}_{inner_key}"] = [res[key][inner_key] for res in results]
    else:
        res_dict[key] = [res[key] for res in results]

In [ ]:
res_df = pd.DataFrame.from_dict(res_dict)

In [ ]:
res_dict = {}
for key in results_base[0].keys():
    if key in ["prediction"]:
        res_dict[key] = [res[key][0] if key in res.keys() else "" for res in results_base]
    elif key in ["estimations"]:
        for inner_key in results_base[0][key].keys():
            if inner_key not in ["matching_scores", 'max_matching_score', 'min_matching_score', 'minmax_matching_score']:
                res_dict[f"{key}_{inner_key}"] = [res[key][inner_key] for res in results_base]
    else:
        res_dict[key] = [res[key] for res in results_base]

In [ ]:
res_df_base = pd.DataFrame.from_dict(res_dict)

In [ ]:
res_df

In [ ]:
res_df_base

In [ ]:
matched_predictions_ids = np.arange(len(res_df))[np.array(res_df["prediction"] == res_df_base["prediction"])]

In [ ]:
print(len(matched_predictions_ids), len(res_df_base))

In [ ]:
res_df_base.iloc[matched_predictions_ids].to_csv("../med_data_paper/med_diag_max_with_ue_test_old_nli_matched_only.csv", index=False)#[3]

In [ ]:
res_df.iloc[matched_predictions_ids].to_csv("../med_data_paper/med_diag_max_with_ue_test_med_nli_matched_only.csv", index=False)#[3]

## Load new test with UE for BB GigaMax, aligned NLI

In [ ]:
log_file_med_nli = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/workdir/output/qa/Sber/GigaChat-Max-V2/AIRI-NLP/med_f/2026-02-12/13-38-33/polygraph_eval.log'
# giga-max, BB, base NLI, some new setting - check if work
log_file_base_nli = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/workdir/output/qa/Sber/GigaChat-Max-V2/AIRI-NLP/med_f/2026-02-12/19-04-42/polygraph_eval.log'
# [DEBUG ONLY], paper res uncommmented. check mathes for initial v1 exps - where UE normal, but generations are strange
#log_file_med_nli = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/workdir/output/qa/Sber/GigaChat-Max-V2/AIRI-NLP/med_f/2026-02-12/08-03-17/polygraph_eval.log'
# giga-max, BB, base NLI, some new setting - check if work
#log_file_base_nli = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/workdir/output/qa/Sber/GigaChat-Max-V2/AIRI-NLP/med_f/2026-02-12/10-15-38/polygraph_eval.log'

# new results with aligned NLI models
log_file_base_nli = '/home/jovyan/gkuzmin/llm_ue/med_data/bb_with_aligned_nli_methods/polygraph_eval.log'


with open(log_file_med_nli, 'r', encoding='utf-8') as f:
    log_content = f.read()

results = parse_log_file_no_target(log_content)

with open(log_file_base_nli, 'r', encoding='utf-8') as f:
    log_content = f.read()

results_base = parse_log_file_no_target(log_content)

In [ ]:
print(log_content[:40000])

In [ ]:
res_dict = {}
for key in results[0].keys():
    if key in ["prediction"]:
        res_dict[key] = [res[key][0] if key in res.keys() else "" for res in results]
    elif key in ["estimations"]:
        for inner_key in results[0][key].keys():
            if inner_key not in ["matching_scores", 'max_matching_score', 'min_matching_score', 'minmax_matching_score']:
                res_dict[f"{key}_{inner_key}"] = [res[key][inner_key] for res in results]
    else:
        res_dict[key] = [res[key] for res in results]

In [ ]:
res_df = pd.DataFrame.from_dict(res_dict)

In [ ]:
res_dict = {}
for key in results_base[0].keys():
    if key in ["prediction"]:
        res_dict[key] = [res[key][0] if key in res.keys() else "" for res in results_base]
    elif key in ["estimations"]:
        for inner_key in results_base[0][key].keys():
            if inner_key not in ["matching_scores", 'max_matching_score', 'min_matching_score', 'minmax_matching_score']:
                res_dict[f"{key}_{inner_key}"] = [res[key][inner_key] for res in results_base]
    else:
        res_dict[key] = [res[key] for res in results_base]

In [ ]:
res_df_base = pd.DataFrame.from_dict(res_dict)

In [ ]:
#res_df

In [ ]:
res_df_base

In [ ]:
res_df_base.to_csv("../med_data_paper/med_diag_max_with_ue_test_both_nli_aligned.csv", index=False)#[3]

In [ ]:
#res_df.iloc[matched_predictions_ids].to_csv("../med_data_paper/med_diag_max_with_ue_test_med_nli_matched_only.csv", index=False)#[3]

# GigaLite, UE with new NLI model, build csv

In [ ]:
# todo: split each answer on claims (and target)
# match each answer claim with target

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df_prev = pd.read_csv(path)

In [ ]:
df_prev["target"].iloc[1]

In [ ]:
path = base_file_path + "med_diag_lite_with_ue_test_new_nli.csv"# "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
transform_target = lambda x: x.strip("[]'").replace("', '", ", ")
transform_greedy = lambda x: x.strip("'")

In [ ]:
df["target"] = df["target"].map(transform_target)
df["prediction"] = df["greedy_text"].map(transform_greedy)

In [ ]:
df

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
with open(base_file_path + "artifacts/pair-match.json", 'r', encoding='utf-8') as f:
    matcher_data = json.load(f)
medical_matcher = MedicalMatcherClaim(matcher_data)

In [ ]:
df.iloc[idx][["prediction", "target"]]

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
# load tokenizer from approx. model
tok = AutoTokenizer.from_pretrained("/home/jovyan/rvanova/giga1/Lite_v2/hf")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:-1] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        claim_tokens = tok(str(text_claim))["input_ids"][1:-1]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_lite_fixed_format_with_claims_intro_fix_new_nli.csv", index=False)

# GigaLite, UE with old NLI model fix + new NLI on CE, build csv

In [ ]:
# todo: split each answer on claims (and target)
# match each answer claim with target

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df_prev = pd.read_csv(path)

In [ ]:
df_prev["target"].iloc[1]

In [ ]:
path = base_file_path + "med_diag_lite_with_ue_test_old_nli_fix.csv"# "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
transform_target = lambda x: x.strip("[]'").replace("', '", ", ")
transform_greedy = lambda x: x.strip("'")

In [ ]:
df["target"] = df["target"].map(transform_target)
df["prediction"] = df["greedy_text"].map(transform_greedy)

In [ ]:
df

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
with open(base_file_path + "artifacts/pair-match.json", 'r', encoding='utf-8') as f:
    matcher_data = json.load(f)
medical_matcher = MedicalMatcherClaim(matcher_data)

In [ ]:
df.iloc[idx][["prediction", "target"]]

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
# load tokenizer from approx. model
tok = AutoTokenizer.from_pretrained("/home/jovyan/rvanova/giga1/Lite_v2/hf")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:-1] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        claim_tokens = tok(str(text_claim))["input_ids"][1:-1]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_lite_fixed_format_with_claims_intro_fix_old_nli_fix.csv", index=False)

# GigaMaxBig, UE with old NLI, build csv

In [ ]:
# todo: split each answer on claims (and target)
# match each answer claim with target

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix.csv"
df_prev = pd.read_csv(path)

In [ ]:
df_prev["target"].iloc[1]

In [ ]:
df_prev

In [ ]:
path = base_file_path + "med_diag_max_with_ue_test_old_nli_matched_only.csv"# "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
transform_target = lambda x: x.strip("[]'").replace("', '", ", ")
transform_greedy = lambda x: x.strip("'")

In [ ]:
df["target"] = df["target"].map(transform_target)
df["prediction"] = df["greedy_text"].map(transform_greedy)

In [ ]:
df

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
with open(base_file_path + "artifacts/pair-match.json", 'r', encoding='utf-8') as f:
    matcher_data = json.load(f)
medical_matcher = MedicalMatcherClaim(matcher_data)

In [ ]:
df.iloc[idx][["prediction", "target"]]

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim_new_test(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
# load tokenizer from approx. model
tok = AutoTokenizer.from_pretrained("/home/jovyan/rvanova/giga1/Lite_v2/hf")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:-1] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        claim_tokens = tok(str(text_claim))["input_ids"][1:-1]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix_NEW_test_with_ue_old_nli_v2.csv", index=False)

In [ ]:
len(df)*3

# GigaBig, Prepare claim-level data (orig format) - match with dict, test set (Recalculated test to match with BB methods)

In [ ]:
def preprocess_prediction_fixed_format_new_test(prediction, lite_train_split=False):
    
    if not prediction:
        return []
    
    if isinstance(prediction, list):
        if len(prediction) > 0:
            prediction_text = prediction[0]
        else:
            return []
    else:
        prediction_text = str(prediction)
    if "<diag_diag>" in prediction_text:
        #lines = prediction_text.split('<DIAG>')
        buf_lines = prediction_text.split('<diag_diag>')
        if lite_train_split:
            splitted_lines = [el.split('</diag_diag>')[0] for el in buf_lines if len(el) != 0]
        else:
            splitted_lines = [el.replace('</diag_diag>', "") for el in buf_lines if len(el) != 0]
        lines = []
        for el in splitted_lines:
            lines.append(el.strip(".?!"))
    else:
        lines = prediction_text.split(';')
    diagnoses = []
    diagnoses_orig = []
    
    for line in lines:
        line = line.strip()
        match = re.match(r'^(\d+\.\s*)?([^\(\)\\n]+?)(?=\\n|\n|$|\(|\.\s|$)', line)
        if match:
            diagnosis = match.group(2).strip().lower()
            diagnosis_orig = match.group(2).strip()
            if (len(diagnosis) > 3 and 
                not diagnosis.startswith('эти') and
                not diagnosis.startswith('наиболее') and
                '...' not in diagnosis):
                diagnoses.append(diagnosis)
                diagnoses_orig.append(diagnosis_orig)
    
    return diagnoses[:3], diagnoses_orig[:3]

def calculate_em_claim_new_test(item, matcher):
    
    if 'prediction' not in item or 'target' not in item:
        return 0.0
    #print(item['prediction'])
    #print(item['target'])
    pred_diagnoses, pred_diagnoses_orig = preprocess_prediction_fixed_format_new_test(item['prediction'])
    #print(pred_diagnoses)
    target_diagnoses = preprocess_target_fixed_format(item['target'])#[diag.lower() for diag in item['target']]
    if not pred_diagnoses or not target_diagnoses:
        return np.array([0]), [item['prediction']]
    
    return relaxed_em_matching(pred_diagnoses, target_diagnoses, matcher), pred_diagnoses_orig

In [ ]:
# todo: split each answer on claims (and target)
# match each answer claim with target

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df_prev = pd.read_csv(path)

In [ ]:
df_prev["target"].iloc[1]

In [ ]:
path = base_file_path + "med_diag_max_with_ue_test_new_nli.csv"# "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
transform_target = lambda x: x.strip("[]'").replace("', '", ", ")
transform_greedy = lambda x: x.strip("'")

In [ ]:
df["target"] = df["target"].map(transform_target)
df["prediction"] = df["greedy_text"].map(transform_greedy)

In [ ]:
df

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
with open(base_file_path + "artifacts/pair-match.json", 'r', encoding='utf-8') as f:
    matcher_data = json.load(f)
medical_matcher = MedicalMatcherClaim(matcher_data)

In [ ]:
df.iloc[idx][["prediction", "target"]]

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim_new_test(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
# load tokenizer from approx. model
tok = AutoTokenizer.from_pretrained("/home/jovyan/rvanova/giga1/Lite_v2/hf")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:-1] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        claim_tokens = tok(str(text_claim))["input_ids"][1:-1]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix_NEW_test_with_ue.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + "uhead_datasets/gigachat_big_with_lite/new_test/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.load_from_disk(base_file_path + "uhead_datasets/gigachat_big_with_lite/new_test/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.DatasetDict({"train": buf_dataset, "test": buf_dataset})
buf_dataset.save_to_disk(base_file_path + "uhead_datasets/gigachat_big_with_lite/new_test/med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix_full_test")

In [ ]:
t_labs = [el for el in buf_dataset["test"]["verified"]]
t_labs_flat = []
for el in t_labs:
    t_labs_flat += el
print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

In [ ]:
# patch original hp set with new test, to recalc TB UQ methods
data_with_val = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")

In [ ]:
data_with_val["test"] = buf_dataset["test"]

In [ ]:
data_with_val.save_to_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix_NEW_test")

# GigaBig, Prepare claim-level data (orig format) - match with dict, test set (Recalculated test with matched with another NLI version, final for now)

In [ ]:
def preprocess_prediction_fixed_format_new_test(prediction, lite_train_split=False):
    
    if not prediction:
        return []
    
    if isinstance(prediction, list):
        if len(prediction) > 0:
            prediction_text = prediction[0]
        else:
            return []
    else:
        prediction_text = str(prediction)
    if "<diag_diag>" in prediction_text:
        #lines = prediction_text.split('<DIAG>')
        buf_lines = prediction_text.split('<diag_diag>')
        if lite_train_split:
            splitted_lines = [el.split('</diag_diag>')[0] for el in buf_lines if len(el) != 0]
        else:
            splitted_lines = [el.replace('</diag_diag>', "") for el in buf_lines if len(el) != 0]
        lines = []
        for el in splitted_lines:
            lines.append(el.strip(".?!"))
    else:
        lines = prediction_text.split(';')
    diagnoses = []
    diagnoses_orig = []
    
    for line in lines:
        line = line.strip()
        match = re.match(r'^(\d+\.\s*)?([^\(\)\\n]+?)(?=\\n|\n|$|\(|\.\s|$)', line)
        if match:
            diagnosis = match.group(2).strip().lower()
            diagnosis_orig = match.group(2).strip()
            if (len(diagnosis) > 3 and 
                not diagnosis.startswith('эти') and
                not diagnosis.startswith('наиболее') and
                '...' not in diagnosis):
                diagnoses.append(diagnosis)
                diagnoses_orig.append(diagnosis_orig)
    
    return diagnoses[:3], diagnoses_orig[:3]

def calculate_em_claim_new_test(item, matcher):
    
    if 'prediction' not in item or 'target' not in item:
        return 0.0
    #print(item['prediction'])
    #print(item['target'])
    pred_diagnoses, pred_diagnoses_orig = preprocess_prediction_fixed_format_new_test(item['prediction'])
    #print(pred_diagnoses)
    target_diagnoses = preprocess_target_fixed_format(item['target'])#[diag.lower() for diag in item['target']]
    if not pred_diagnoses or not target_diagnoses:
        return np.array([0]), [item['prediction']]
    
    return relaxed_em_matching(pred_diagnoses, target_diagnoses, matcher), pred_diagnoses_orig

In [ ]:
# todo: split each answer on claims (and target)
# match each answer claim with target

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df_prev = pd.read_csv(path)

In [ ]:
df_prev["target"].iloc[1]

In [ ]:
path = base_file_path + "med_diag_max_with_ue_test_med_nli_matched_only.csv"# "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
transform_target = lambda x: x.strip("[]'").replace("', '", ", ")
transform_greedy = lambda x: x.strip("'")

In [ ]:
df["target"] = df["target"].map(transform_target)
df["prediction"] = df["greedy_text"].map(transform_greedy)

In [ ]:
df

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
with open(base_file_path + "artifacts/pair-match.json", 'r', encoding='utf-8') as f:
    matcher_data = json.load(f)
medical_matcher = MedicalMatcherClaim(matcher_data)

In [ ]:
df.iloc[0][["prediction", "target"]]

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim_new_test(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
# load tokenizer from approx. model
tok = AutoTokenizer.from_pretrained("/home/jovyan/rvanova/giga1/Lite_v2/hf")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:-1] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        claim_tokens = tok(str(text_claim))["input_ids"][1:-1]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix_NEW_test_with_ue_med_nli_v2.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + "uhead_datasets/gigachat_big_with_lite/new_test_v2/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.load_from_disk(base_file_path + "uhead_datasets/gigachat_big_with_lite/new_test_v2/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.DatasetDict({"train": buf_dataset, "test": buf_dataset})
buf_dataset.save_to_disk(base_file_path + "uhead_datasets/gigachat_big_with_lite/new_test_v2/med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix_full_test")

In [ ]:
t_labs = [el for el in buf_dataset["test"]["verified"]]
t_labs_flat = []
for el in t_labs:
    t_labs_flat += el
print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

In [ ]:
# patch original hp set with new test, to recalc TB UQ methods
data_with_val = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")

In [ ]:
data_with_val["test"] = buf_dataset["test"]

In [ ]:
data_with_val.save_to_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix_NEW_test_v2")

# GigaBig, Prepare claim-level data (orig format) - match with dict, test set (With aligned NLI)

In [ ]:
def preprocess_prediction_fixed_format_new_test(prediction, lite_train_split=False):
    
    if not prediction:
        return []
    
    if isinstance(prediction, list):
        if len(prediction) > 0:
            prediction_text = prediction[0]
        else:
            return []
    else:
        prediction_text = str(prediction)
    if "<diag_diag>" in prediction_text:
        #lines = prediction_text.split('<DIAG>')
        buf_lines = prediction_text.split('<diag_diag>')
        if lite_train_split:
            splitted_lines = [el.split('</diag_diag>')[0] for el in buf_lines if len(el) != 0]
        else:
            splitted_lines = [el.replace('</diag_diag>', "") for el in buf_lines if len(el) != 0]
        lines = []
        for el in splitted_lines:
            lines.append(el.strip(".?!"))
    else:
        lines = prediction_text.split(';')
    diagnoses = []
    diagnoses_orig = []
    
    for line in lines:
        line = line.strip()
        match = re.match(r'^(\d+\.\s*)?([^\(\)\\n]+?)(?=\\n|\n|$|\(|\.\s|$)', line)
        if match:
            diagnosis = match.group(2).strip().lower()
            diagnosis_orig = match.group(2).strip()
            if (len(diagnosis) > 3 and 
                not diagnosis.startswith('эти') and
                not diagnosis.startswith('наиболее') and
                '...' not in diagnosis):
                diagnoses.append(diagnosis)
                diagnoses_orig.append(diagnosis_orig)
    
    return diagnoses[:3], diagnoses_orig[:3]

def calculate_em_claim_new_test(item, matcher):
    
    if 'prediction' not in item or 'target' not in item:
        return 0.0
    #print(item['prediction'])
    #print(item['target'])
    pred_diagnoses, pred_diagnoses_orig = preprocess_prediction_fixed_format_new_test(item['prediction'])
    #print(pred_diagnoses)
    target_diagnoses = preprocess_target_fixed_format(item['target'])#[diag.lower() for diag in item['target']]
    if not pred_diagnoses or not target_diagnoses:
        return np.array([0]), [item['prediction']]
    
    return relaxed_em_matching(pred_diagnoses, target_diagnoses, matcher), pred_diagnoses_orig

In [ ]:
# todo: split each answer on claims (and target)
# match each answer claim with target

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df_prev = pd.read_csv(path)

In [ ]:
df_prev["target"].iloc[1]

In [ ]:
path = base_file_path + "med_diag_max_with_ue_test_both_nli_aligned.csv"# "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
transform_target = lambda x: x.strip("[]'").replace("', '", ", ")
transform_greedy = lambda x: x.strip("'")

In [ ]:
df["target"] = df["target"].map(transform_target)
df["prediction"] = df["greedy_text"].map(transform_greedy)

In [ ]:
df

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
with open(base_file_path + "artifacts/pair-match.json", 'r', encoding='utf-8') as f:
    matcher_data = json.load(f)
medical_matcher = MedicalMatcherClaim(matcher_data)

In [ ]:
df.iloc[0][["prediction", "target"]]

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim_new_test(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
# load tokenizer from approx. model
tok = AutoTokenizer.from_pretrained("/home/jovyan/rvanova/giga1/Lite_v2/hf")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:-1] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        if idx == 6:
            text_claim = " " + text_claim
        claim_tokens = tok(str(text_claim))["input_ids"][1:-1]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix_NEW_test_with_ue_med_nli_v2_with_aligned_nli.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + "uhead_datasets/gigachat_big_with_lite/new_test_v2_aligned_nli/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.load_from_disk(base_file_path + "uhead_datasets/gigachat_big_with_lite/new_test_v2_aligned_nli/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.DatasetDict({"train": buf_dataset, "test": buf_dataset})
buf_dataset.save_to_disk(base_file_path + "uhead_datasets/gigachat_big_with_lite/new_test_v2_aligned_nli/med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix_full_test")

In [ ]:
t_labs = [el for el in buf_dataset["test"]["verified"]]
t_labs_flat = []
for el in t_labs:
    t_labs_flat += el
print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

In [ ]:
# patch original hp set with new test, to recalc TB UQ methods
data_with_val = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")

In [ ]:
data_with_val["test"] = buf_dataset["test"]

In [ ]:
data_with_val.save_to_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix_NEW_test_v2_aligned_nli")

# GigaLite, Prepare claim-level data (orig format) - match with dict, test set

In [ ]:
# todo: split each answer on claims (and target)
# match each answer claim with target

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df_prev = pd.read_csv(path)

In [ ]:
df_prev["target"].iloc[1]

In [ ]:
path = base_file_path + "med_diag_lite_with_ue_test.csv"# "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
transform_target = lambda x: x.strip("[]'").replace("', '", ", ")
transform_greedy = lambda x: x.strip("'")

In [ ]:
df["target"] = df["target"].map(transform_target)
df["prediction"] = df["greedy_text"].map(transform_greedy)

In [ ]:
df

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
with open(base_file_path + "artifacts/pair-match.json", 'r', encoding='utf-8') as f:
    matcher_data = json.load(f)
medical_matcher = MedicalMatcherClaim(matcher_data)

In [ ]:
df.iloc[idx][["prediction", "target"]]

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
# load tokenizer from approx. model
tok = AutoTokenizer.from_pretrained("/home/jovyan/rvanova/giga1/Lite_v2/hf")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:-1] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        claim_tokens = tok(str(text_claim))["input_ids"][1:-1]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_lite_fixed_format_with_claims_intro_fix.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + "uhead_datasets/gigachat_lite/raw_med_data_giga_lite_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.load_from_disk(base_file_path + "uhead_datasets/gigachat_lite/raw_med_data_giga_lite_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.DatasetDict({"train": buf_dataset, "test": buf_dataset.select([0])})
buf_dataset.save_to_disk(base_file_path + "uhead_datasets/gigachat_lite/med_data_giga_lite_relaxed_em_target_claim_fixed_format_intro_fix_full_train")

In [ ]:
x, y = np.arange(len(new_dataset)), [el[0] for el in new_dataset["verified"]]
skf = KFold(n_splits=5, random_state=None)
skf.get_n_splits(x)
for i, (train, test) in enumerate(skf.split(x, y)):
    print(f"Fold {i}:")
    print(f"  Train:  index={train}")
    print(f"  Test:  index={test}")
    print(list(test))
    med_dataset = datasets.DatasetDict({
        "train": new_dataset.select(train),
        "test": new_dataset.select(test),
    })
    med_dataset.save_to_disk(base_file_path + f"uhead_datasets/gigachat_lite/med_data_giga_lite_relaxed_em_target_claim_fixed_format_intro_fix_cv_{i}")
#train, test = train_test_split(x, test_size=0.2, stratify=y, random_state=35)


In [ ]:
# inspect dataset stats - estimate base performance scores

In [ ]:
for idx in range(5):
    buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_lite/med_data_giga_lite_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    t_labs = [el for el in buf_med_data["test"]["verified"]]
    t_labs_flat = []
    for el in t_labs:
        t_labs_flat += el
    print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
    print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

## Do the same for the small holdout set

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_holdout_set_intro_fix.csv"
prev_df = pd.read_csv(path)

In [ ]:
path = base_file_path + "med_diag_lite_with_ue_holdout.csv"
df = pd.read_csv(path)

In [ ]:
df["target"].iloc[4].strip("[]'").replace("', '", ", ").replace("'\n '", ", ")

In [ ]:
df

In [ ]:
transform_target = lambda x: x.strip("[]'").replace("', '", ", ").replace("'\n '", ", ")
transform_greedy = lambda x: x.strip("'")

In [ ]:
df["target"] = df["target"].map(transform_target)
df["prediction"] = df["greedy_text"].map(transform_greedy)

In [ ]:
df

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
nli_model_path = base_file_path + "artifacts/distilled-dm-model"
medical_matcher = MedicalMatcherClaimNeural(nli_model_path)

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
tok = AutoTokenizer.from_pretrained("/home/jovyan/rvanova/giga1/Lite_v2/hf")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:-1] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        claim_tokens = tok(str(text_claim))["input_ids"][1:-1]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_lite_fixed_format_holdout_set_with_claims_neural_intro_fix.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + f"uhead_datasets/gigachat_lite/raw_med_data_giga_lite_relaxed_em_target_claim_fixed_format_holdout_set_neural_intro_fix")

In [ ]:
new_dataset_v1 = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_lite/raw_med_data_giga_lite_relaxed_em_target_claim_fixed_format_holdout_set_neural_intro_fix")
new_dataset_v2 = datasets.DatasetDict({"train": new_dataset_v1, "test": new_dataset_v1})
new_dataset_v2.save_to_disk(base_file_path + f"uhead_datasets/gigachat_lite/raw_med_data_giga_lite_relaxed_em_target_claim_fixed_format_holdout_set_neural_intro_fix_test")

# Prepare claim-level data for 1.5k for big model (orig format)

In [ ]:
path = base_file_path + "medical_data_ordered_max_1500_giga_big_fixed_format_intro_fix.csv"
prev_df = pd.read_csv(path)

In [ ]:
prev_df

In [ ]:
prev_df["target"].iloc[0]

In [ ]:
path = base_file_path + "med_diag_lite_with_ue_train.csv"#"medical_data_ordered_max_1500_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
transform_target = lambda x: x.strip("[]'").replace("', '", ", ").replace("\\\\n", "\n")
transform_greedy = lambda x: x.strip("'")

In [ ]:
df["target"] = df["target"].map(transform_target)
df["prediction"] = df["greedy_text"].map(transform_greedy)

In [ ]:
df["target"].iloc[2]

In [ ]:
df

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
nli_model_path = base_file_path + "artifacts/distilled-dm-model"
medical_matcher = MedicalMatcherClaimNeural(nli_model_path)

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim_enlarged_train(df.iloc[idx][["prediction", "target"]], medical_matcher, lite_train_split=True)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
tok = AutoTokenizer.from_pretrained("/home/jovyan/rvanova/giga1/Lite_v2/hf")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:-1] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        if "1. " in new_data["reply"][idx]:
            claim_tokens = tok(" " + str(text_claim))["input_ids"][1:-1]            
        else:
            claim_tokens = tok(str(text_claim))["input_ids"][1:-1]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
#tok.decode([806, 35107, 420, 2391, 6657, 55508, 11377, 4]), tok.decode([806, 35107, 420, 2391, 6657, 55508, 1505, 28884, 98363, 32, 11377, 4])

In [ ]:
#df.iloc[19]["claims"]

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_lite_enlarged_train_fixed_format_with_claims_intro_fix.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + f"uhead_datasets/gigachat_lite/raw_med_data_relaxed_em_target_claim_enlarged_train_fixed_format_intro_fix")

In [ ]:
new_dataset = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_lite/raw_med_data_relaxed_em_target_claim_enlarged_train_fixed_format_intro_fix")

In [ ]:
for idx in range(5):
    med_dataset_test = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_lite/med_data_giga_lite_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    med_dataset_test["train"] = datasets.concatenate_datasets([new_dataset, med_dataset_test["train"]])
    med_dataset_test.save_to_disk(base_file_path + f"uhead_datasets/gigachat_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")

In [ ]:
# split on the same train and test, as before (for med match)

In [ ]:
buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_0")

In [ ]:
for idx in range(5):
    buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    t_labs = [el for el in buf_med_data["test"]["verified"]]
    t_labs_flat = []
    for el in t_labs:
        t_labs_flat += el
    #print(t_labs, el)
    print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
    print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

In [ ]:
for idx in range(5):
    buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    t_labs = [el for el in buf_med_data["train"]["verified"]]
    t_labs_flat = []
    for el in t_labs:
        t_labs_flat += el
    print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
    print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

### Prepare dataset for hp - test - all 260 samples, other - train and valid

In [ ]:
new_dataset = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_lite/raw_med_data_relaxed_em_target_claim_enlarged_train_fixed_format_intro_fix")

In [ ]:
for idx in range(5):
    med_dataset_test = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_lite/med_data_giga_lite_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    if idx == 0:
        buf_test = med_dataset_test["test"]
    else:
        buf_test = datasets.concatenate_datasets([buf_test, med_dataset_test["test"]])

In [ ]:
buf_test

In [ ]:
# split train on train and val
train_ids = np.arange(len(new_dataset))
train_ids, val_ids = train_test_split(train_ids, test_size=0.1, random_state=43)

In [ ]:
len(train_ids), len(val_ids)

In [ ]:
data_with_val = {
    "train": new_dataset.select(train_ids),
    "validation": new_dataset.select(val_ids),
    "test": buf_test,
}
data_with_val = datasets.DatasetDict(data_with_val)

In [ ]:
data_with_val.save_to_disk(base_file_path + f"uhead_datasets/gigachat_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")

# Extract examples from mock set

In [ ]:
mock_data = pd.read_csv("../med_data_paper/output_with_claims_with_uhead_ue.csv")

In [ ]:
idx = 10
print(mock_data.iloc[idx]["input"], "\n", mock_data.iloc[idx]["output"])

### Now do the same, but for picture

In [ ]:
mock_data = pd.read_csv("../med_data_paper/output_with_claims_with_uhead_ue.csv")

In [ ]:
idx = 17#11
print(mock_data.iloc[idx]["input"], "\n", mock_data.iloc[idx]["output"], "\n", mock_data.iloc[idx]["ue_scores"])

In [ ]:

class MedicalMatcherClaimNeuralDebug:
    def __init__(self, nli_model_path="./med_data/distilled-dm-model"):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(nli_model_path)
        model = AutoModelForSequenceClassification.from_pretrained(nli_model_path)
        model.to(device)
        model.eval()
        self.device = device
        self.model = model
        self.tokenizer = tokenizer
    
    def match(self, diagnosis1, diagnosis2):
        diagnosis1 = diagnosis1.lower()
        diagnosis2 = diagnosis2.lower()
        
        key1 = f"{diagnosis1}|{diagnosis2}"
        key2 = f"{diagnosis2}|{diagnosis1}"
        inputs = self.tokenizer(
            diagnosis1, diagnosis2,
            padding=True, truncation=True, max_length=128,
            return_tensors="pt"
        ).to(self.device)
        all_probs = []
        with torch.no_grad():
            logits = self.model(**inputs).logits
            probs = torch.softmax(logits, dim=1)
            print(probs)
            all_probs.extend(probs[:, 1].cpu().tolist())
        print(all_probs)
        y_scores = np.array(all_probs)
        # print(diagnosis1, diagnosis2, y_scores)
        return 1 if y_scores[0] > 0.5 else 0


In [ ]:
nli_model_path = base_file_path + "artifacts/distilled-dm-model"
medical_matcher_debug = MedicalMatcherClaimNeuralDebug(nli_model_path)

In [ ]:
from sentence_transformers import CrossEncoder

model = CrossEncoder('cross-encoder/stsb-roberta-large')#CrossEncoder(base_file_path + "artifacts/distilled-dm-model")
#scores = model.predict([('Sentence 1', 'Sentence 2'), ('Sentence 3', 'Sentence 4')])

In [ ]:
scores = model.predict([('Sentence 1', 'Sentence 2'), ('Sentence 3', 'Sentence 4'), ('Sentence 5', 'Sentence 6'), ("Несахарный диабет", "Несахарный диабет")])

In [ ]:
scores

In [ ]:
from scipy.special import softmax

In [ ]:
softmax(scores, axis=-1)[:,-1]

In [ ]:
medical_matcher_debug.match("Несахарный диабет", "Несахарный диабет")

In [ ]:
t11 = datasets.load_dataset("AIRI-NLP/med_train_f")

In [ ]:
t11

# [GIGA2 20B approx Open-Source] GigaMax, approximation with small open Giga3

# GigaBig, Prepare claim-level data (orig format) - match with dict, test set

In [ ]:
# todo: split each answer on claims (and target)
# match each answer claim with target

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
with open(base_file_path + "artifacts/pair-match.json", 'r', encoding='utf-8') as f:
    matcher_data = json.load(f)
medical_matcher = MedicalMatcherClaim(matcher_data)

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
# load tokenizer from approx. model
tok = AutoTokenizer.from_pretrained("ai-sage/GigaChat-20B-A3B-instruct-v1.5-bf16")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        claim_tokens = tok(str(text_claim))["input_ids"][1:]#[1:-1]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "giga2_20b_approx_medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + "uhead_datasets/giga2_20b_approx_gigachat_big/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.load_from_disk(base_file_path + "uhead_datasets/giga2_20b_approx_gigachat_big/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.DatasetDict({"train": buf_dataset, "test": buf_dataset.select([0])})
buf_dataset.save_to_disk(base_file_path + "uhead_datasets/giga2_20b_approx_gigachat_big/med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix_full_train")

In [ ]:
x, y = np.arange(len(new_dataset)), [el[0] for el in new_dataset["verified"]]
skf = KFold(n_splits=5, random_state=None)
skf.get_n_splits(x)
for i, (train, test) in enumerate(skf.split(x, y)):
    print(f"Fold {i}:")
    print(f"  Train:  index={train}")
    print(f"  Test:  index={test}")
    print(list(test))
    med_dataset = datasets.DatasetDict({
        "train": new_dataset.select(train),
        "test": new_dataset.select(test),
    })
    med_dataset.save_to_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix_cv_{i}")
#train, test = train_test_split(x, test_size=0.2, stratify=y, random_state=35)


In [ ]:
# inspect dataset stats - estimate base performance scores

In [ ]:
for idx in range(5):
    buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    t_labs = [el for el in buf_med_data["test"]["verified"]]
    t_labs_flat = []
    for el in t_labs:
        t_labs_flat += el
    print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
    print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

## Do the same for the small holdout set

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_holdout_set_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
nli_model_path = base_file_path + "artifacts/distilled-dm-model"
medical_matcher = MedicalMatcherClaimNeural(nli_model_path)

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
tok = AutoTokenizer.from_pretrained("ai-sage/GigaChat-20B-A3B-instruct-v1.5-bf16")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        claim_tokens = tok(str(text_claim))["input_ids"][1:]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "giga2_20b_approx_medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims_neural_intro_fix.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_holdout_set_neural_intro_fix")

In [ ]:
new_dataset_v1 = datasets.load_from_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_holdout_set_neural_intro_fix")
new_dataset_v2 = datasets.DatasetDict({"train": new_dataset_v1, "test": new_dataset_v1})
new_dataset_v2.save_to_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_holdout_set_neural_intro_fix_test")

# Prepare claim-level data for 1.5k for big model (orig format)

In [ ]:
path = base_file_path + "medical_data_ordered_max_1500_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
nli_model_path = base_file_path + "artifacts/distilled-dm-model"
medical_matcher = MedicalMatcherClaimNeural(nli_model_path)

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim_enlarged_train(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
tok = AutoTokenizer.from_pretrained("ai-sage/GigaChat-20B-A3B-instruct-v1.5-bf16")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        if "1. " in new_data["reply"][idx]:
            claim_tokens = tok(" " + str(text_claim))["input_ids"][1:]            
        else:
            claim_tokens = tok(str(text_claim))["input_ids"][1:]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "giga2_20b_approx_medical_data_ordered_giga_big_enlarged_train_fixed_format_with_claims_intro_fix.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/raw_med_data_relaxed_em_target_claim_enlarged_train_fixed_format_intro_fix")

In [ ]:
new_dataset = datasets.load_from_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/raw_med_data_relaxed_em_target_claim_enlarged_train_fixed_format_intro_fix")

In [ ]:
for idx in range(5):
    med_dataset_test = datasets.load_from_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    med_dataset_test["train"] = datasets.concatenate_datasets([new_dataset, med_dataset_test["train"]])
    med_dataset_test.save_to_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")

In [ ]:
# split on the same train and test, as before (for med match)

In [ ]:
buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_0")

In [ ]:
for idx in range(5):
    buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    t_labs = [el for el in buf_med_data["test"]["verified"]]
    t_labs_flat = []
    for el in t_labs:
        t_labs_flat += el
    #print(t_labs, el)
    print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
    print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

In [ ]:
for idx in range(5):
    buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    t_labs = [el for el in buf_med_data["train"]["verified"]]
    t_labs_flat = []
    for el in t_labs:
        t_labs_flat += el
    print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
    print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

### Prepare dataset for hp - test - all 260 samples, other - train and valid

In [ ]:
new_dataset = datasets.load_from_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/raw_med_data_relaxed_em_target_claim_enlarged_train_fixed_format_intro_fix")

In [ ]:
for idx in range(5):
    med_dataset_test = datasets.load_from_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    if idx == 0:
        buf_test = med_dataset_test["test"]
    else:
        buf_test = datasets.concatenate_datasets([buf_test, med_dataset_test["test"]])

In [ ]:
buf_test

In [ ]:
# split train on train and val
train_ids = np.arange(len(new_dataset))
train_ids, val_ids = train_test_split(train_ids, test_size=0.1, random_state=43)

In [ ]:
len(train_ids), len(val_ids)

In [ ]:
data_with_val = {
    "train": new_dataset.select(train_ids),
    "validation": new_dataset.select(val_ids),
    "test": buf_test,
}
data_with_val = datasets.DatasetDict(data_with_val)

In [ ]:
data_with_val.save_to_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")

# GigaBig, Prepare claim-level data (orig format) - match with dict, test set (Recalculated test to match with BB methods)

In [ ]:
def preprocess_prediction_fixed_format_new_test(prediction, lite_train_split=False):
    
    if not prediction:
        return []
    
    if isinstance(prediction, list):
        if len(prediction) > 0:
            prediction_text = prediction[0]
        else:
            return []
    else:
        prediction_text = str(prediction)
    if "<diag_diag>" in prediction_text:
        #lines = prediction_text.split('<DIAG>')
        buf_lines = prediction_text.split('<diag_diag>')
        if lite_train_split:
            splitted_lines = [el.split('</diag_diag>')[0] for el in buf_lines if len(el) != 0]
        else:
            splitted_lines = [el.replace('</diag_diag>', "") for el in buf_lines if len(el) != 0]
        lines = []
        for el in splitted_lines:
            lines.append(el.strip(".?!"))
    else:
        lines = prediction_text.split(';')
    diagnoses = []
    diagnoses_orig = []
    
    for line in lines:
        line = line.strip()
        match = re.match(r'^(\d+\.\s*)?([^\(\)\\n]+?)(?=\\n|\n|$|\(|\.\s|$)', line)
        if match:
            diagnosis = match.group(2).strip().lower()
            diagnosis_orig = match.group(2).strip()
            if (len(diagnosis) > 3 and 
                not diagnosis.startswith('эти') and
                not diagnosis.startswith('наиболее') and
                '...' not in diagnosis):
                diagnoses.append(diagnosis)
                diagnoses_orig.append(diagnosis_orig)
    
    return diagnoses[:3], diagnoses_orig[:3]

def calculate_em_claim_new_test(item, matcher):
    
    if 'prediction' not in item or 'target' not in item:
        return 0.0
    #print(item['prediction'])
    #print(item['target'])
    pred_diagnoses, pred_diagnoses_orig = preprocess_prediction_fixed_format_new_test(item['prediction'])
    #print(pred_diagnoses)
    target_diagnoses = preprocess_target_fixed_format(item['target'])#[diag.lower() for diag in item['target']]
    if not pred_diagnoses or not target_diagnoses:
        return np.array([0]), [item['prediction']]
    
    return relaxed_em_matching(pred_diagnoses, target_diagnoses, matcher), pred_diagnoses_orig

In [ ]:
# todo: split each answer on claims (and target)
# match each answer claim with target

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df_prev = pd.read_csv(path)

In [ ]:
df_prev["target"].iloc[1]

In [ ]:
path = base_file_path + "med_diag_max_with_ue_test_new_nli.csv"# "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
transform_target = lambda x: x.strip("[]'").replace("', '", ", ")
transform_greedy = lambda x: x.strip("'")

In [ ]:
df["target"] = df["target"].map(transform_target)
df["prediction"] = df["greedy_text"].map(transform_greedy)

In [ ]:
df

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
with open(base_file_path + "artifacts/pair-match.json", 'r', encoding='utf-8') as f:
    matcher_data = json.load(f)
medical_matcher = MedicalMatcherClaim(matcher_data)

In [ ]:
df.iloc[idx][["prediction", "target"]]

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim_new_test(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
# load tokenizer from approx. model
tok = AutoTokenizer.from_pretrained("ai-sage/GigaChat-20B-A3B-instruct-v1.5-bf16")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        claim_tokens = tok(str(text_claim))["input_ids"][1:]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "giga2_20b_approx_medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix_NEW_test_with_ue.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + "uhead_datasets/giga2_20b_approx_gigachat_big/new_test/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.load_from_disk(base_file_path + "uhead_datasets/giga2_20b_approx_gigachat_big/new_test/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.DatasetDict({"train": buf_dataset, "test": buf_dataset})
buf_dataset.save_to_disk(base_file_path + "uhead_datasets/giga2_20b_approx_gigachat_big/new_test/med_data_giga_big_relaxed_em_target_claim_fixed_format_intro_fix_full_test")

In [ ]:
t_labs = [el for el in buf_dataset["test"]["verified"]]
t_labs_flat = []
for el in t_labs:
    t_labs_flat += el
print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

In [ ]:
# patch original hp set with new test, to recalc TB UQ methods
data_with_val = datasets.load_from_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")

In [ ]:
data_with_val["test"] = buf_dataset["test"]

In [ ]:
data_with_val.save_to_disk(base_file_path + f"uhead_datasets/giga2_20b_approx_gigachat_big/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix_NEW_test")

# [GIGA2 20B - claim] Prepare claim-level data in the original format - now with fixed demographic

In [ ]:
def parse_log_file_no_target(log_content: str) -> List[Dict]:
    results = []
    current_batch = {}
    current_estimations = {}
    current_matching_scores = []  
    in_estimations = False
    in_generation_metrics = False

    input_re = re.compile(r"input_texts: \['(.*?)'\]")
    target_re = re.compile(r"target_texts: \['(.*?)'\]")
    greedy_re = re.compile(r"greedy_texts: \['(.*?)'\]")
    estimator_re = re.compile(r"\('claim', '(.*?)'\): \[(.*?)\]")
    
    result_re = re.compile(r"RESULT: (?:True|False) \(([\d.]+)\)")
    prev_line = ""

    for line in log_content.split('\n'):
        if "================================================== NEW BATCH ==================================================" in line:
            
            if current_batch:
                
                if current_matching_scores:
                    
                    current_estimations['matching_scores'] = current_matching_scores
                    current_estimations['max_matching_score'] = max(current_matching_scores)
                    current_estimations['min_matching_score'] = min(current_matching_scores)
                    current_estimations['minmax_matching_score'] = 1 - (max(current_matching_scores) - min(current_matching_scores))
                    #medical_score = calculate_medical_score(current_batch, matcher)
                    #current_estimations['medical_match'] = medical_score
                
                if current_estimations:
                    current_batch['estimations'] = current_estimations
                
                results.append(current_batch)
            
            
            current_batch = {}
            current_estimations = {}
            current_matching_scores = []  
            in_estimations = False
            in_generation_metrics = False
            continue  

        
        if "RESULT:" in line:
            match = result_re.search(line)
            if match:
                score = float(match.group(1))
                current_matching_scores.append(score)

        if "input_texts" in line:
            match = input_re.search(line)
            if match:
                current_batch['input'] = match.group(1)

        if "target_texts" in line:
            match = target_re.search(line)
            if match:
                current_batch['target'] = [d.strip() for d in match.group(1).split(',') if d.strip()]

        if "greedy_texts" in line:
            match = greedy_re.search(line)
            greedy_text = line.split("greedy_texts:")[1].split("[")[1].split("]")[0]
            current_batch['greedy_text'] = greedy_text
            if match:
                current_batch['prediction'] = [d.strip() for d in match.group(1).split(',') if d.strip()]

        if "Estimations:" in line:
            in_estimations = True
            in_generation_metrics = False
            continue

        if "Generation metrics:" in line:
            in_estimations = False
            in_generation_metrics = True
            continue

        if (in_estimations or in_generation_metrics) and line.strip():
            match = estimator_re.search(line)
            if match:
                estimator_name = match.group(1)
                value_str = match.group(2)
                try:
                    #if ',' in value_str:
                    values = [float(v) for v in value_str.split(',')]
                    #value = values[0] if values else 0.0
                    #else:
                    #    value = float(value_str)
                    current_estimations[estimator_name] = values
                except ValueError:
                    continue
        prev_line = line

    
    if current_batch:
        
        if current_matching_scores:
            current_estimations['matching_scores'] = current_matching_scores
            current_estimations['max_matching_score'] = max(current_matching_scores)
            current_estimations['min_matching_score'] = min(current_matching_scores)
            current_estimations['minmax_matching_score'] = 1 - (max(current_matching_scores) - min(current_matching_scores))
        
            medical_score = calculate_medical_score(current_batch, matcher)
            current_estimations['medical_match'] = medical_score
        
        if current_estimations:
            current_batch['estimations'] = current_estimations
        
        results.append(current_batch)

    return results

In [ ]:
# test set
log_file = '/home/jovyan/gkuzmin/llm_ue/med_ue_polygraph/med_polygraph/workdir/output/qa/home/jovyan/rvanova/giga1/Lite_v2/hf/AIRI-NLP/med_f/2026-02-06/14-57-55/polygraph_eval.log'
# holdout set
#log_file = '/home/jovyan/gkuzmin/llm_ue/med_ue_polygraph/med_polygraph/workdir/output/qa/home/jovyan/rvanova/giga1/Lite_v2/hf/AIRI-NLP/med_f/2026-02-06/16-39-14/polygraph_eval.log'
# train set
#log_file = '/home/jovyan/gkuzmin/llm_ue/med_ue_polygraph/med_polygraph/workdir/output/qa/home/jovyan/rvanova/giga1/Lite_v2/hf/AIRI-NLP/med_train_f/2026-02-06/17-03-57/polygraph_eval.log'
# test set with new NLI model
log_file = '/home/jovyan/gkuzmin/llm_ue/med_ue_polygraph/med_polygraph/workdir/output/qa/home/jovyan/rvanova/giga1/Lite_v2/hf/AIRI-NLP/med_f/2026-02-10/15-33-52/polygraph_eval.log'
# test set with old NLI model fix + new NLI on crossencoder
log_file = '/home/jovyan/gkuzmin/llm_ue/med_ue_polygraph/med_polygraph/workdir/output/qa/home/jovyan/rvanova/giga1/Lite_v2/hf/AIRI-NLP/med_f/2026-02-10/18-58-41/polygraph_eval.log'
# giga-max, BB, base NLI
log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/workdir/output/qa/Sber/GigaChat-Max-V2/AIRI-NLP/med_f/2026-02-12/10-15-38/polygraph_eval.log'
# giga-max, BB, new med NLI, some new setting - check if work
log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/workdir/output/qa/Sber/GigaChat-Max-V2/AIRI-NLP/med_f/2026-02-12/13-38-33/polygraph_eval.log'
# giga-max, BB, base NLI, some new setting - check if work
# giga2-20b, WB, test set, med NLI
log_file = '/home/jovyan/gkuzmin/llm_ue/med_ue_polygraph/med_polygraph/workdir/output/qa/ai-sage/GigaChat-20B-A3B-instruct-v1.5-bf16/AIRI-NLP/med_f/2026-02-25/20-33-49/polygraph_eval.log'
# giga2-20b, WB, test set, base NLI
log_file = '/home/jovyan/gkuzmin/llm_ue/med_ue_polygraph/med_polygraph/workdir/output/qa/ai-sage/GigaChat-20B-A3B-instruct-v1.5-bf16/AIRI-NLP/med_f/2026-02-26/03-33-39/polygraph_eval.log'
# giga2-20b, WB, train set, med NLI
log_file = '/home/jovyan/gkuzmin/llm_ue/med_ue_polygraph/med_polygraph/workdir/output/qa/ai-sage/GigaChat-20B-A3B-instruct-v1.5-bf16/AIRI-NLP/med_train_f/2026-02-27/20-20-55/polygraph_eval.log'



with open(log_file, 'r', encoding='utf-8') as f:
    log_content = f.read()

results = parse_log_file_no_target(log_content)

In [ ]:
results[0]

In [ ]:
res_dict = {}
for key in results[0].keys():
    if key in ["prediction"]:
        res_dict[key] = [res[key][0] if key in res.keys() else "" for res in results]
    elif key in ["estimations"]:
        for inner_key in results[0][key].keys():
            if inner_key not in ["matching_scores", 'max_matching_score', 'min_matching_score', 'minmax_matching_score']:
                res_dict[f"{key}_{inner_key}"] = [res[key][inner_key] for res in results]
    else:
        res_dict[key] = [res[key] for res in results]

In [ ]:
for idx, res in enumerate(results):
    #if idx == 18:
    #    print(res["prediction"])
    if "prediction" not in res.keys():
        #print(res)
        #print(res.keys())
        print(idx)

In [ ]:
res_df = pd.DataFrame.from_dict(res_dict)

In [ ]:
import copy
bse_nli_res_df = copy.deepcopy(res_df)

In [ ]:
#np.array(res_df["prediction"] == bse_nli_res_df["prediction"])

In [ ]:
res_df.head(20)

In [ ]:
#res_df.to_csv("../med_data_paper/med_diag_lite_with_ue_test_old_nli_fix.csv", index=False)#[3]
#res_df.to_csv("../med_data_paper/med_diag_max_with_ue_test_new_nli.csv", index=False)#[3]
res_df.to_csv("../med_data_paper/med_diag_giga2_20b_with_ue_train_new_nli.csv", index=False)#[3]

# Giga2-20B, Prepare claim-level data (orig format) - match with dict, test set

In [ ]:
# todo: split each answer on claims (and target)
# match each answer claim with target

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df_prev = pd.read_csv(path)

In [ ]:
df_prev["target"].iloc[1]

In [ ]:
path = base_file_path + "med_diag_giga2_20b_with_ue_test_new_nli.csv"# "medical_data_ordered_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
transform_target = lambda x: x.strip("[]'").replace("', '", ", ")
transform_greedy = lambda x: x.strip("'")

In [ ]:
df["target"] = df["target"].map(transform_target)
df["prediction"] = df["greedy_text"].map(transform_greedy)

In [ ]:
df

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
with open(base_file_path + "artifacts/pair-match.json", 'r', encoding='utf-8') as f:
    matcher_data = json.load(f)
medical_matcher = MedicalMatcherClaim(matcher_data)

In [ ]:
df.iloc[0][["prediction", "target"]]

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim(df.iloc[idx][["prediction", "target"]], medical_matcher, lite_train_split=True)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
# load tokenizer from approx. model
tok = AutoTokenizer.from_pretrained("ai-sage/GigaChat-20B-A3B-instruct-v1.5-bf16")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        claim_tokens = tok(str(text_claim))["input_ids"][1:]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0 and f" {text_claim}" in new_data["reply"][idx]:
            # match claim with first space
            claim_tokens = tok(str(" " + text_claim))["input_ids"][1:]
            matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_2_20b_fixed_format_with_claims_intro_fix.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + "uhead_datasets/gigachat_2_20b/raw_med_data_giga_lite_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.load_from_disk(base_file_path + "uhead_datasets/gigachat_2_20b/raw_med_data_giga_lite_relaxed_em_target_claim_fixed_format_intro_fix")

In [ ]:
buf_dataset = datasets.DatasetDict({"train": buf_dataset, "test": buf_dataset.select([0])})
buf_dataset.save_to_disk(base_file_path + "uhead_datasets/gigachat_2_20b/med_data_giga_lite_relaxed_em_target_claim_fixed_format_intro_fix_full_train")

In [ ]:
x, y = np.arange(len(new_dataset)), [el[0] for el in new_dataset["verified"]]
skf = KFold(n_splits=5, random_state=None)
skf.get_n_splits(x)
for i, (train, test) in enumerate(skf.split(x, y)):
    print(f"Fold {i}:")
    print(f"  Train:  index={train}")
    print(f"  Test:  index={test}")
    print(list(test))
    med_dataset = datasets.DatasetDict({
        "train": new_dataset.select(train),
        "test": new_dataset.select(test),
    })
    med_dataset.save_to_disk(base_file_path + f"uhead_datasets/gigachat_2_20b/med_data_giga_lite_relaxed_em_target_claim_fixed_format_intro_fix_cv_{i}")
#train, test = train_test_split(x, test_size=0.2, stratify=y, random_state=35)


In [ ]:
# inspect dataset stats - estimate base performance scores

In [ ]:
for idx in range(5):
    buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_2_20b/med_data_giga_lite_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    t_labs = [el for el in buf_med_data["test"]["verified"]]
    t_labs_flat = []
    for el in t_labs:
        t_labs_flat += el
    print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
    print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

## Do the same for the small holdout set [DO NOT USED]

In [ ]:
path = base_file_path + "medical_data_ordered_giga_big_fixed_format_holdout_set_intro_fix.csv"
prev_df = pd.read_csv(path)

In [ ]:
path = base_file_path + "med_diag_lite_with_ue_holdout.csv"
df = pd.read_csv(path)

In [ ]:
df["target"].iloc[4].strip("[]'").replace("', '", ", ").replace("'\n '", ", ")

In [ ]:
df

In [ ]:
transform_target = lambda x: x.strip("[]'").replace("', '", ", ").replace("'\n '", ", ")
transform_greedy = lambda x: x.strip("'")

In [ ]:
df["target"] = df["target"].map(transform_target)
df["prediction"] = df["greedy_text"].map(transform_greedy)

In [ ]:
df

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
nli_model_path = base_file_path + "artifacts/distilled-dm-model"
medical_matcher = MedicalMatcherClaimNeural(nli_model_path)

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim(df.iloc[idx][["prediction", "target"]], medical_matcher)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
tok = AutoTokenizer.from_pretrained("/home/jovyan/rvanova/giga1/Lite_v2/hf")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:-1] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        claim_tokens = tok(str(text_claim))["input_ids"][1:-1]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(text_claim)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_lite_fixed_format_holdout_set_with_claims_neural_intro_fix.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + f"uhead_datasets/gigachat_lite/raw_med_data_giga_lite_relaxed_em_target_claim_fixed_format_holdout_set_neural_intro_fix")

In [ ]:
new_dataset_v1 = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_lite/raw_med_data_giga_lite_relaxed_em_target_claim_fixed_format_holdout_set_neural_intro_fix")
new_dataset_v2 = datasets.DatasetDict({"train": new_dataset_v1, "test": new_dataset_v1})
new_dataset_v2.save_to_disk(base_file_path + f"uhead_datasets/gigachat_lite/raw_med_data_giga_lite_relaxed_em_target_claim_fixed_format_holdout_set_neural_intro_fix_test")

# Prepare claim-level data for 1.5k for big model (orig format)

In [ ]:
path = base_file_path + "medical_data_ordered_max_1500_giga_big_fixed_format_intro_fix.csv"
prev_df = pd.read_csv(path)

In [ ]:
prev_df

In [ ]:
prev_df["target"].iloc[0]

In [ ]:
path = base_file_path + "med_diag_giga2_20b_with_ue_train_new_nli.csv"#"medical_data_ordered_max_1500_giga_big_fixed_format_intro_fix.csv"
df = pd.read_csv(path)

In [ ]:
df

In [ ]:
transform_target = lambda x: x.strip("[]'").replace("', '", ", ").replace("\\\\n", "\n")
transform_greedy = lambda x: x.strip("'")

In [ ]:
df["target"] = df["target"].map(transform_target)
df["prediction"] = df["greedy_text"].map(transform_greedy)

In [ ]:
cols_to_use = ['input', 'target', 'prediction']
mapping = ['question', 'answer', 'reply']
# to add
# 'input_ids', 'claims', 'greedy_ids', 'uncertainty_labels_original', 'uncertainty_labels', 'uncertainty_labels', 'verified', 'alignscore'

In [ ]:
nli_model_path = base_file_path + "artifacts/distilled-dm-model"
medical_matcher = MedicalMatcherClaimNeural(nli_model_path)

In [ ]:
claims_is_true = []
claims = []
for idx in range(len(df)):
    claim_is_true, claim = calculate_em_claim_enlarged_train(df.iloc[idx][["prediction", "target"]], medical_matcher, lite_train_split=True)
    claims_is_true.append(claim_is_true)
    claims.append(claim)

In [ ]:
df["claims"] = claims
df["claims_is_true"] = claims_is_true

In [ ]:
df["claims"].iloc[45]

In [ ]:
tok = AutoTokenizer.from_pretrained("ai-sage/GigaChat-20B-A3B-instruct-v1.5-bf16")

In [ ]:
new_data = {}
new_data["question"] = [el for el in df['input']]
new_data["answer"] = [el for el in df['target']]
new_data["reply"] = [str(el) for el in df['prediction']]

new_data["input_ids"] = [tok(el1 + str(el2))["input_ids"] for el1, el2 in zip(df['input'], df['prediction'])]
new_data["greedy_ids"] = [tok(str(el2))["input_ids"][1:] for el2 in df['prediction']]

In [ ]:
# match claims to tokens and after set uncertainty labels
new_data["claims"] = []
new_data["verified"] = []
#for idx in range(len(df["claims_is_true"])):
#    buf = df.iloc[idx]["claims_is_true"]
#    verified = [int(1-el) for el in buf]
#    new_data["verified"].append(verified)
unmatch_count = 0
for idx, el in enumerate(new_data["greedy_ids"]):
    buf_claim_data = []
    buf_verified_data = []
    buf_claims = df.iloc[idx]["claims"]
    buf_verified = df.iloc[idx]["claims_is_true"]
    prev_matched_tok_start = 0
    for inner_idx, text_claim in enumerate(buf_claims):
        # strip bos and eos
        claim_tokens = None
        #if "1. " in new_data["reply"][idx]:
        #    claim_tokens = tok(" " + str(text_claim))["input_ids"][1:]
        #else:
        claim_tokens = tok(str(text_claim))["input_ids"][1:]
        #if claim_tokens is None and new_data["reply"][idx].startswith(str(text_claim) + "\\n"):
        #    claim_tokens = tok(text_claim + "\\")["input_ids"][1:-1]
        matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0 and f" {text_claim}" in new_data["reply"][idx]:
            # match claim with first space
            claim_tokens = tok(str(" " + text_claim))["input_ids"][1:]
            matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
        if len(matched_toks) == 0:
            print(f"=========={idx}============")
            print("=========Unmatched tokens==============")
            unmatch_count += 1
            print(el)
            print(claim_tokens)
            print(new_data["reply"][idx])
            print(matched_toks)
        else:
            matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
            prev_matched_tok_start = matched_toks[0]+1
            buf_claim_data.append({"aligned_token_ids": matched_toks,
                                   "claim_text": str(text_claim),
                                   "sentence": new_data["reply"][idx]})
            buf_verified_data.append(int(1-buf_verified[inner_idx]))
    new_data["claims"].append(buf_claim_data)
    new_data["verified"].append(buf_verified_data)
print(unmatch_count)

In [ ]:
tok.decode([66448, 2903, 18625]), tok.decode([95317, 2903, 18625])

In [ ]:
#tok.decode([806, 35107, 420, 2391, 6657, 55508, 11377, 4]), tok.decode([806, 35107, 420, 2391, 6657, 55508, 1505, 28884, 98363, 32, 11377, 4])

In [ ]:
#df.iloc[19]["claims"]

In [ ]:
verified_inverse = [[int(1-x) for x in el] for el in new_data["verified"]]
df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]
df.to_csv(base_file_path + "medical_data_ordered_giga_2_20b_enlarged_train_fixed_format_with_claims_intro_fix.csv", index=False)

In [ ]:
new_data["uncertainty_labels"] = []
for idx, el in enumerate(new_data["input_ids"]):
    buf = np.ones_like(el) * -100
    sample_shift = len(el) - len(new_data["greedy_ids"][idx])
    for claim, verif in zip(new_data["claims"][idx], new_data["verified"][idx]):
        tokens_ids = [el + sample_shift for el in claim["aligned_token_ids"]]
        buf[tokens_ids] = verif
    #label = new_data["verified"][idx][0]
    #buf[-1*len(new_data["greedy_ids"][idx]):] = label
    new_data["uncertainty_labels"].append(buf)
new_data["uncertainty_labels_original"] = new_data["uncertainty_labels"]


In [ ]:
new_dataset = datasets.Dataset.from_dict(new_data)
new_dataset.save_to_disk(base_file_path + f"uhead_datasets/gigachat_2_20b/raw_med_data_relaxed_em_target_claim_enlarged_train_fixed_format_intro_fix")

In [ ]:
new_dataset = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_2_20b/raw_med_data_relaxed_em_target_claim_enlarged_train_fixed_format_intro_fix")

In [ ]:
for idx in range(5):
    med_dataset_test = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_2_20b/med_data_giga_lite_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    med_dataset_test["train"] = datasets.concatenate_datasets([new_dataset, med_dataset_test["train"]])
    med_dataset_test.save_to_disk(base_file_path + f"uhead_datasets/gigachat_2_20b/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")

In [ ]:
# split on the same train and test, as before (for med match)

In [ ]:
buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_2_20b/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_0")

In [ ]:
for idx in range(5):
    buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_2_20b/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    t_labs = [el for el in buf_med_data["test"]["verified"]]
    t_labs_flat = []
    for el in t_labs:
        t_labs_flat += el
    #print(t_labs, el)
    print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
    print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

In [ ]:
for idx in range(5):
    buf_med_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_2_20b/med_data_1_5k_full_7b_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    t_labs = [el for el in buf_med_data["train"]["verified"]]
    t_labs_flat = []
    for el in t_labs:
        t_labs_flat += el
    print(np.unique(t_labs_flat, return_counts=True), len(t_labs_flat))
    print(np.unique(t_labs_flat, return_counts=True)[1][0] / np.sum(np.unique(t_labs_flat, return_counts=True)[1]))

### Prepare dataset for hp - test - all 260 samples, other - train and valid

In [ ]:
new_dataset = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_2_20b/raw_med_data_relaxed_em_target_claim_enlarged_train_fixed_format_intro_fix")

In [ ]:
for idx in range(5):
    med_dataset_test = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_2_20b/med_data_giga_lite_relaxed_em_target_claim_fixed_format_intro_fix_cv_{idx}")
    if idx == 0:
        buf_test = med_dataset_test["test"]
    else:
        buf_test = datasets.concatenate_datasets([buf_test, med_dataset_test["test"]])

In [ ]:
buf_test

In [ ]:
# split train on train and val
train_ids = np.arange(len(new_dataset))
train_ids, val_ids = train_test_split(train_ids, test_size=0.1, random_state=43)

In [ ]:
len(train_ids), len(val_ids)

In [ ]:
data_with_val = {
    "train": new_dataset.select(train_ids),
    "validation": new_dataset.select(val_ids),
    "test": buf_test,
}
data_with_val = datasets.DatasetDict(data_with_val)

In [ ]:
data_with_val.save_to_disk(base_file_path + f"uhead_datasets/gigachat_2_20b/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")